In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- portfolio_stock_from_pandas ---
FIX_PORTFOLIO_STOCK_FROM_PANDAS_INTERVAL = "1H"
FIX_PORTFOLIO_STOCK_FROM_PANDAS_NAME = "BRCA1"
FIX_PORTFOLIO_STOCK_FROM_PANDAS_PERIOD = 14
_PORTFOLIO_HISTORY_PD = pd.DataFrame(
    {"Close": [100.0, 101.0, 102.0], "Volume": [10, 12, 11]},
    index=pd.date_range("2023-01-01", periods=3, freq="D", name="Datetime"),
)
class _MockTicker:
    def __init__(self, name):
        self.name = name
    def history(self, period=None, interval=None):
        return _PORTFOLIO_HISTORY_PD.copy()
FIX_PORTFOLIO_STOCK_FROM_PANDAS_YF = SimpleNamespace(Ticker=_MockTicker)

# --- portfolio_stock_to_dict ---
FIX_PORTFOLIO_STOCK_TO_DICT_D = {}

print("✅ Fixtures loaded")
DF_STOCK_PD = pd.DataFrame({"Datetime": pd.date_range("2023-01-01", periods=5, freq="D"), "Close": [100.0,101.0,102.0,101.5,103.0]})
DF_STOCK_PL = pl.from_pandas(DF_STOCK_PD)
df = DF_STOCK_PD


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_portfolio_stock_from_pandas(interval, name, period, yf):
    ticker = yf.Ticker(name)
    df = ticker.history(period=period, interval=interval)
    df['Datetime'] = df.index
    return df
    return df

def before_portfolio_stock_to_dict(d):
    for d in df.to_dict('records'):
        pass
    return None

In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_portfolio_stock_from_pandas(interval, name, period, yf):
    ticker = yf.Ticker(name)
    df = ticker.history(period=period, interval=interval)
    df = pl.from_pandas(df.reset_index()).rename({"index": "Datetime"}) if "index" in df.reset_index().columns else pl.from_pandas(df.reset_index())
    return df
    return df

def gen_portfolio_stock_to_dict(d):

    for d in df.to_dicts():
        pass
    return None

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: portfolio_stock_to_dict ===
import sys

def _capture_last_record(func, frame, arg):
    global df
    old_df = df
    captured = {}
    target = func.__code__
    def tracer(frame_obj, event, value):
        if frame_obj.f_code is target and event == "return":
            captured.update(frame_obj.f_locals)
        return tracer
    try:
        df = frame
        sys.settrace(tracer)
        result = func(arg)
    finally:
        sys.settrace(None)
        df = old_df
    return result, captured.get("d", object())

try:
    _r, _ = _capture_last_record(gen_portfolio_stock_to_dict, DF_STOCK_PL, FIX_PORTFOLIO_STOCK_TO_DICT_D)
    print("✅ L1 smoke gen_portfolio_stock_to_dict: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_portfolio_stock_to_dict: {type(_e).__name__}: {_e}")

try:
    _rb, _ = _capture_last_record(before_portfolio_stock_to_dict, DF_STOCK_PD, FIX_PORTFOLIO_STOCK_TO_DICT_D)
    print("✅ L1 smoke before_portfolio_stock_to_dict: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_portfolio_stock_to_dict: {type(_e).__name__}: {_e}")

try:
    _, _before_last = _capture_last_record(before_portfolio_stock_to_dict, DF_STOCK_PD, FIX_PORTFOLIO_STOCK_TO_DICT_D)
    _, _gen_last = _capture_last_record(gen_portfolio_stock_to_dict, DF_STOCK_PL, FIX_PORTFOLIO_STOCK_TO_DICT_D)
    if _before_last == _gen_last:
        print("✅ L2 equivalence portfolio_stock_to_dict records: MATCH")
    else:
        print(f"❌ L2 equivalence portfolio_stock_to_dict records: MISMATCH — before={_before_last!r}, gen={_gen_last!r}")
except Exception as _e:
    print(f"❌ L2 equivalence portfolio_stock_to_dict: setup error — {type(_e).__name__}: {_e}")

try:
    _, _before_last = _capture_last_record(before_portfolio_stock_to_dict, DF_STOCK_PD.head(0), FIX_PORTFOLIO_STOCK_TO_DICT_D)
    _, _gen_last = _capture_last_record(gen_portfolio_stock_to_dict, DF_STOCK_PL.head(0), FIX_PORTFOLIO_STOCK_TO_DICT_D)
    if _before_last == _gen_last:
        print("✅ L3 edge portfolio_stock_to_dict empty df: MATCH")
    else:
        print(f"❌ L3 edge portfolio_stock_to_dict empty df: MISMATCH — before={_before_last!r}, gen={_gen_last!r}")
except Exception as _e:
    print(f"❌ L3 edge portfolio_stock_to_dict: {type(_e).__name__}: {_e}")
